# 03. 어텐티브 프로브와 절제 결과 읽기

목표: 단순 평균 풀링과 학습 가능한 질의 기반 어텐티브 풀링을 비교하고, 논문의 표를 과장 없이 읽는 분석 습관을 익힙니다.

In [ ]:
from math import exp, sqrt

tokens = [[0.2, 0.1, 0.0], [0.1, 0.0, 0.2], [2.0, 1.8, 0.1], [0.0, 0.1, 0.2]]
query = [1.0, 1.0, 0.0]  # 행동 단서를 담은 세 번째 토큰을 선호하는 학습된 질의를 가정합니다.

def average_pool(rows):
    return [sum(values) / len(rows) for values in zip(*rows)]

def attentive_pool(rows, query_vector):
    scale = sqrt(len(query_vector))
    scores = [sum(a * b for a, b in zip(row, query_vector)) / scale for row in rows]
    maximum = max(scores)
    unnormalized = [exp(score - maximum) for score in scores]
    weights = [value / sum(unnormalized) for value in unnormalized]
    pooled = [sum(weight * row[index] for weight, row in zip(weights, rows))
              for index in range(len(query_vector))]
    return pooled, weights

attentive, weights = attentive_pool(tokens, query)
print('평균 풀링:', [round(value, 3) for value in average_pool(tokens)])
print('어텐티브 풀링:', [round(value, 3) for value in attentive])
print('토큰별 가중치:', [round(value, 3) for value in weights])
assert weights[2] == max(weights)

In [ ]:
# 논문 표 4의 세 다운스트림 지표입니다. 평균은 실습용 보조 지표일 뿐 논문의 공식 종합 점수가 아닙니다.
masking = {
    'random-tube': {'K400': 51.5, 'SSv2': 46.4, 'IN1K': 55.6},
    'causal-6': {'K400': 61.3, 'SSv2': 49.8, 'IN1K': 66.9},
    'causal-12': {'K400': 71.9, 'SSv2': 63.6, 'IN1K': 72.2},
    'multi-block': {'K400': 72.9, 'SSv2': 67.4, 'IN1K': 72.8},
}
ranking = sorted(
    ((sum(scores.values()) / len(scores), name) for name, scores in masking.items()),
    reverse=True,
)
for average, name in ranking:
    print(f'{name:12s} 실습용 단순 평균={average:.2f}  원지표={masking[name]}')
assert ranking[0][1] == 'multi-block'

In [ ]:
# 서로 다른 평가 예산은 분리해서 읽어야 합니다.
temporal_coverage = {'1 clip': 73.7, '8 clips': 80.9}
label_efficiency_h16 = {
    'K400': {'5% labels': 68.2, '100% linear probe': 82.0},
    'SSv2': {'5% labels': 54.0, '100% linear probe': 71.4},
}
print('K400에서 8개 clip 사용 시 차이:', temporal_coverage['8 clips'] - temporal_coverage['1 clip'])
for dataset, values in label_efficiency_h16.items():
    ratio = values['5% labels'] / values['100% linear probe']
    print(f'{dataset}: 5% 라벨 점수/전체 선형 프로브 점수 = {ratio:.1%}')

print('주의: clip 수, probe 구조, 해상도가 다른 숫자를 모델 품질 하나로 단순 비교하면 안 됩니다.')

## 다음 실험

질의 벡터를 바꾸어 어떤 토큰에 가중치가 몰리는지 확인하고, 각 마스킹 방식의 데이터셋별 순위를 따로 계산해 보세요. 실제 재현에서는 clip 수, 공간 해상도, probe 깊이, fine-tuning 여부를 반드시 같은 조건으로 맞춰야 합니다.